# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.  

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and fields
print("Available Record Sets:")
record_set_overview = []
for record_set in dataset.metadata.record_sets:
    print(f"- Record Set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name:30} (@id: {field.id})  dataType: {field.data_type}")
    print('')
    record_set_overview.append(record_set.id)

# For detailed inspection, show an example record from the first record set (change as needed)
if record_set_overview:
    print(f"Sample record from {record_set_overview[0]}:")
    for i, row in enumerate(dataset.records(record_set=record_set_overview[0])):
        pprint.pprint(row)
        if i > 0:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from available record sets into DataFrames
import warnings
warnings.filterwarnings('ignore')

# Use the record set IDs found above
record_sets = record_set_overview.copy()
dataframes = {}

for recset_id in record_sets:
    print(f"Loading record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f" - Columns in {recset_id}: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print(f" - No records found in {recset_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filter, normalize, and group a numerical field
if record_sets:
    # Select the first record set and try to find a quantitative field
    main_record_set = record_sets[0]
    df = dataframes[main_record_set]

    # Heuristically find a numeric field (float/int) and a categorical field
    numeric_field = None
    group_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            if numeric_field is None:
                numeric_field = col
        else:
            if group_field is None:
                group_field = col

    print(f"Using '{numeric_field}' for numeric analysis.")
    print(f"Using '{group_field}' for grouping if available.")

    # Example: Filter values above (mean + 1 std)
    if numeric_field and numeric_field in df.columns:
        threshold = df[numeric_field].mean() + df[numeric_field].std()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} ({filtered_df.shape[0]} records):")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized values:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group if possible
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_sets and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded and explored the FAIR²-compliant Croissant dataset on mass spectrometry analysis of extracellular vesicles from human mast cells. Using `mlcroissant`, we programmatically listed record sets and their fields (referenced by `@id`), loaded tabular data, filtered and normalized quantitative fields, and visualized major trends. This workflow can be adapted to other FAIR² datasets described by Croissant schemas by referencing entity `@id`s and using the `mlcroissant` API for data-centric analysis.*